# Update a MODFLOW WEL file with FloPy

This notebook shows how to load a MODFLOW-2005 WEL file, update pumping rates, and write an MF6 WEL package using FloPy.


In [ ]:
# If needed, install flopy in this environment
# !pip install flopy


## 1) Download the WEL file


In [ ]:
import urllib.request

url = (
    "https://ckan.tacc.utexas.edu/dataset/18400624-423c-42b5-ad56-6c73322584bd/"
    "resource/9c7b25c4-8cea-4965-a07a-d9b3867f18a9/"
    "download/barton_springs_2001_2010average.wel"
)
wel_path = "barton_springs_2001_2010average.wel"
urllib.request.urlretrieve(url, wel_path)
print("Downloaded", wel_path)


## 2) Load the WEL file with FloPy (MODFLOW-2005)

We need a minimal DIS to load the WEL. We can scan the file to infer max row/col/layer.


In [ ]:
from pathlib import Path
import flopy

def scan_wel_metadata(path):
    def strip_comment(line):
        for token in ("#", ";"):
            if token in line:
                line = line.split(token, 1)[0]
        return line.strip()

    lines = [strip_comment(line) for line in Path(path).read_text().splitlines()]
    data_lines = [line for line in lines if line]
    if not data_lines:
        raise ValueError("WEL file is empty or has no data.")
    data_lines.pop(0)  # header

    nper = 0
    max_k = max_i = max_j = 1
    idx = 0
    while idx < len(data_lines):
        tokens = data_lines[idx].split()
        idx += 1
        if not tokens:
            continue
        nper += 1
        itmp = int(tokens[0])
        if itmp <= 0:
            continue
        for _ in range(itmp):
            if idx >= len(data_lines):
                raise ValueError("Unexpected end of file while scanning wells.")
            parts = data_lines[idx].split()
            idx += 1
            k, i, j = (int(parts[0]), int(parts[1]), int(parts[2]))
            max_k = max(max_k, k)
            max_i = max(max_i, i)
            max_j = max(max_j, j)

    return nper, max_k, max_i, max_j

nper, nlay, nrow, ncol = scan_wel_metadata(wel_path)
print("nper, nlay, nrow, ncol =", nper, nlay, nrow, ncol)

m = flopy.modflow.Modflow(modelname="wel_read", model_ws=".")
flopy.modflow.ModflowDis(
    m,
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    nper=nper,
    delr=1.0,
    delc=1.0,
    top=1.0,
    botm=[0.0] * nlay,
)
wel = flopy.modflow.ModflowWel.load(wel_path, m)
wel


## 3) Update pumping rates (example: scale by 10%)


In [ ]:
import numpy as np

spd = wel.stress_period_data.data

# Inspect one stress period
first_per = spd[0]
print(first_per.dtype.names)
print(first_per[:3])

# Scale all pumping rates by 1.1
for per, recs in spd.items():
    if len(recs) == 0:
        continue
    recs["flux"] *= 1.1

wel.stress_period_data = spd
print("Updated rates")


## 4) Write an MF6 WEL package with FloPy


In [ ]:
# Convert MF2005 WEL recarray -> MF6 stress period data
mf6_spd = {}
for per, recs in spd.items():
    items = []
    for rec in recs:
        k = int(rec["k"]) - 1
        i = int(rec["i"]) - 1
        j = int(rec["j"]) - 1
        q = float(rec["flux"])
        items.append(((k, i, j), q))
    mf6_spd[per] = items

sim = flopy.mf6.MFSimulation(sim_name="wel_update", version="mf6", sim_ws=".")
flopy.mf6.ModflowTdis(
    sim,
    time_units="DAYS",
    nper=nper,
    perioddata=[(1.0, 1, 1.0)] * nper,
)
gwf = flopy.mf6.ModflowGwf(sim, modelname="gwf")
flopy.mf6.ModflowGwfdis(
    gwf,
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    delr=1.0,
    delc=1.0,
    top=1.0,
    botm=[0.0] * nlay,
)
wel6 = flopy.mf6.ModflowGwfwel(
    gwf,
    stress_period_data=mf6_spd,
    filename="barton_springs_mf6.wel",
)
wel6.write()
print("Wrote", wel6.filename)


# Use GeoJSON to build a WEL file

This section shows how to map GeoJSON point features to model cells and create MF6 WEL stress period data.
You will need a model grid definition (origin, spacing, rotation, CRS).


In [ ]:
# If needed, install geospatial deps
# !pip install geopandas shapely pyproj


In [ ]:
import geopandas as gpd
import flopy
from shapely.geometry import Point
from flopy.discretization import StructuredGrid
from flopy.utils.gridintersect import GridIntersect

# --- inputs ---
geojson_path = "wells.geojson"
rate_field = "q"      # GeoJSON property holding pumping rate
layer_field = "k"     # GeoJSON property holding layer (1-based)
sp_field = None         # Optional stress period field, e.g. 'per'

# --- grid definition (fill with your model values) ---
nlay, nrow, ncol = 1, 200, 200
delr, delc = 500.0, 500.0
xoff, yoff = 0.0, 0.0
angrot = 0.0
crs = "EPSG:26914"  # change to your model CRS

grid = StructuredGrid(
    nlay=nlay, nrow=nrow, ncol=ncol,
    delr=delr, delc=delc, xoff=xoff, yoff=yoff, angrot=angrot,
)
gdf = gpd.read_file(geojson_path)
if gdf.crs is None:
    raise ValueError("GeoJSON has no CRS; set gdf.crs before reprojecting.")
if gdf.crs != crs:
    gdf = gdf.to_crs(crs)

gi = GridIntersect(grid, method="vertex")

def point_to_cell(point):
    result = gi.intersect(point)
    if result.empty:
        return None
    row = int(result["row"].iloc[0])
    col = int(result["col"].iloc[0])
    return row, col

# Build stress period data
spd = {}
for _, feat in gdf.iterrows():
    if feat.geometry is None or feat.geometry.is_empty:
        continue
    if not isinstance(feat.geometry, Point):
        raise ValueError("GeoJSON must contain point features for wells.")

    cell = point_to_cell(feat.geometry)
    if cell is None:
        continue
    row, col = cell

    k = int(feat[layer_field]) - 1  # convert to 0-based for MF6
    q = float(feat[rate_field])
    per = int(feat[sp_field]) if sp_field else 0

    spd.setdefault(per, []).append(((k, row, col), q))

print("Stress periods:", sorted(spd.keys()))
print("Sample record:", spd[min(spd.keys())][0])


In [ ]:
# Write MF6 WEL package from GeoJSON-driven spd
sim = flopy.mf6.MFSimulation(sim_name="geojson_wel", version="mf6", sim_ws=".")
flopy.mf6.ModflowTdis(
    sim,
    time_units="DAYS",
    nper=max(spd.keys()) + 1 if spd else 1,
    perioddata=[(1.0, 1, 1.0)] * (max(spd.keys()) + 1 if spd else 1),
)
gwf = flopy.mf6.ModflowGwf(sim, modelname="gwf")
flopy.mf6.ModflowGwfdis(
    gwf,
    nlay=nlay, nrow=nrow, ncol=ncol,
    delr=delr, delc=delc, top=1.0, botm=[0.0] * nlay,
)
wel6 = flopy.mf6.ModflowGwfwel(
    gwf,
    stress_period_data=spd,
    filename="wells_from_geojson.wel",
)
wel6.write()
print("Wrote", wel6.filename)
